In [7]:
import json
import pandas as pd
import os
from datetime import datetime, timedelta

# Load JSON
json_path = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/contests.json"
with open(json_path, "r") as f:
    contests = json.load(f)

# Flatten nested metadata

df_contests = pd.DataFrame(contests)

#print(df_contests.head().iloc[1]['metadata'])
#df_contests.iloc[-1]['metadata']={'num_captions': None, 'num_votes': None}
#print(df_contests.iloc[-1:]['metadata'])

# Display the first few columns
#df_contests.head()
print(df_contests.iloc[250:300]['metadata'].keys())

RangeIndex(start=250, stop=300, step=1)


Now we identify the missing metadata 


In [2]:
def missing_meta(meta):
    # Return True if metadata is missing (None or not a dictionary)
    return not isinstance(meta, dict)

# Filter the DataFrame for rows with missing metadata
missing_df = df_contests[df_contests["metadata"].apply(missing_meta)]

# Show results
print(f"Contests missing metadata: {len(missing_df)}\n")
print(missing_df[["contest_id", "data", "metadata"]].head(20))


Contests missing metadata: 144

     contest_id          data metadata
16          527  data/527.csv     None
52          564  data/564.csv     None
76          588  data/588.csv     None
85          597  data/597.csv     None
93          605  data/605.csv     None
103         615  data/615.csv     None
132         644  data/644.csv     None
135         647  data/647.csv     None
144         656  data/656.csv     None
145         657  data/657.csv     None
158         670  data/670.csv     None
202         714  data/714.csv     None
252         764  data/764.csv     None
253         765  data/765.csv     None
254         766  data/766.csv     None
255         767  data/767.csv     None
256         768  data/768.csv     None
257         769  data/769.csv     None
258         770  data/770.csv     None
259         771  data/771.csv     None


Now let's add the missing metadata

In [3]:

csv_folder = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/data"

def update_metadata_from_csv_1(df, base_path=csv_folder):
    for i, row in df.iterrows():
        # Ensure metadata is a dict
        meta = row.get("metadata")
        print(meta)
        if not isinstance(meta, dict):
            meta = {}

        # Skip if both fields already exist
        need_caps = ("num_captions" not in meta) #or (meta.get("num_captions") in [None, ""])
        need_votes = ("num_votes" not in meta) #or (meta.get("num_votes") in [None, ""])
        if not (need_caps or need_votes):
            df.at[i, "metadata"] = meta
            continue

        # Build CSV path from the 'data' field
        data_field = row.get("data")
        if not isinstance(data_field, str) :
            df.at[i, "metadata"] = meta
            print(i)
            continue

        filename = os.path.basename(data_field)      # e.g., "510.csv"
        print(filename)
        csv_path = os.path.join(base_path, filename) # full path to CSV
        print(csv_path)
        # If file not found, skip it
        if not os.path.exists(csv_path):
            df.at[i, "metadata"] = meta
            continue

        print(i)
        # Read CSV and compute values
        try:
            cdf = pd.read_csv(csv_path)
            meta["num_captions"] = int(len(cdf))
            meta["num_votes"] = int(cdf["votes"].sum(skipna=True))
        except Exception as e:
            print(e)
            pass  # just skip if there’s any issue

        # Write metadata dict back into the DataFrame
        df.at[i, "metadata"] = meta
    return df


# Example usage:
df_contests_1 = update_metadata_from_csv_1(df_contests)
#print(df_contests_1.head().iloc[1]['metadata'])
# View a few rows to confirm it worked
#f_contests_1[["contest_id", "metadata"]]
#print(df_contests_1.head().iloc[250,:]['metadata'])


{'num_captions': 3905, 'num_votes': 41185, 'image_locations': ['the street'], 'image_descriptions': ['A man is relaxing on a city street. Others are going about their business and stepping on him as they pass.'], 'image_uncanny_descriptions': ['A man is just laying in the middle of the sidewalk.'], 'entities': ['https://en.wikipedia.org/wiki/Bystander_effect', 'https://en.wikipedia.org/wiki/Crowd'], 'questions': ['Why is he laying there?']}
{'num_captions': 3325, 'num_votes': 28205, 'image_locations': ['the front hard', 'a residential walkway'], 'image_descriptions': ['A man in a winter coat and cap is looking at a small bearded man holding a snow shovel much taller than him. The ground is covered in snow, except for a path leading to a house.', 'A man is in the snow. A boy with a huge snow shovel is there too. They are outside a house.', 'A man is walking down a sidewalk. He is looking at another person hiding in the snow.'], 'image_uncanny_descriptions': ["It's unusual to see someone

In [14]:


#print(df_contests_1.iloc['metadata'])
df_contests_1.to_csv("/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/contests_updated_metadata.csv", index=False)

## Adding missing dates

In [15]:

#print(contests_df['contest_id'])
#print(df_contests['metadata'].iloc[1].keys())

contest_510_publish=datetime.strptime('2016.02.22',"%Y.%m.%d")
contest_510_votes=datetime.strptime('2016.02.29',"%Y.%m.%d")

for i in range(len(df_contests_1)):
    contest_id=df_contests_1['contest_id'].iloc[i]
    #print(contest_id)
    diff=int(contest_id)-510
    contest_day_publish=contest_510_publish+timedelta(days=diff*7)
    #print(contest_day_publish)
    df_contests_1['metadata'].iloc[i]['contest_day_publish']=contest_day_publish ## bug prcq pas de dico/missing values
    df_contests_1['metadata'].iloc[i]['contest_day_votes']=contest_day_publish+timedelta(days=7)
#print(df_contests_1['metadata']['contest_day_publish'])


df_contests_1.iloc[250:300]

,contest_id,image,data,metadata
250,762,images/762.jpg,data/762.csv,"{'num_captions': 7076, 'num_votes': 901611, 'i..."
251,763,images/763.jpg,data/763.csv,"{'num_captions': 9007, 'num_votes': 889556, 'i..."
252,764,images/764.jpg,data/764.csv,"{'num_captions': 6310, 'num_votes': 665368, 'c..."
253,765,images/765.jpg,data/765.csv,"{'num_captions': 6932, 'num_votes': 763962, 'c..."
254,766,images/766.jpg,data/766.csv,"{'num_captions': 5608, 'num_votes': 731238, 'c..."
255,767,images/767.jpg,data/767.csv,"{'num_captions': 7912, 'num_votes': 768653, 'c..."
256,768,images/768.jpg,data/768.csv,"{'num_captions': 6526, 'num_votes': 779519, 'c..."
257,769,images/769.jpg,data/769.csv,"{'num_captions': 8039, 'num_votes': 921031, 'c..."
258,770,images/770.jpg,data/770.csv,"{'num_captions': 5353, 'num_votes': 531879, 'c..."
259,771,images/771.jpg,data/771.csv,"{'num_captions': 6861, 'num_votes': 616562, 'c..."


In [16]:
df_contests_1.to_csv("/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/contests_updated_metadata2.csv", index=False)